Evaluation

In [ ]:
#Accuracy, Precision, Recall, F1-Score

In [ ]:
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

model_columns = {
    "Llama-3.1-70B": "LLM_Topic_llama",
    "GPT-5.1": "LLM_Topic_gpt",
    "Gemini-2.5-Flash": "LLM_Topic_gemini",
    "Qwen3-8B": "Qwen_topic",
    "MiniLM": "SEBRT_miniML",
    "E5-Large": "SBERT_e5Large",
    "BGE-M3": "SBERT_bGE"
}

unassigned = "UNASSIGNED"

def normalize_labels(series):
    return series.astype("string").str.strip().str.upper()

human = normalize_labels(df_human["Human_Topic"])
valid_human = human.notna()

y_true = (human[valid_human] != unassigned).astype(int)

results = []

for model, column in model_columns.items():

    predictions = normalize_labels(
        df_human.loc[valid_human, column]
    )

    # Do not silently treat missing predictions as valid labels
    valid = predictions.notna()

    y = y_true[valid]
    pred = (predictions[valid] != unassigned).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    results.append({
        "Model": model,
        "N": len(y),
        "Assigned (%)": 100 * pred.mean(),
        "Precision": precision_score(
            y, pred, zero_division=0
        ),
        "Recall": recall_score(
            y, pred, zero_division=0
        ),
        "F1": f1_score(
            y, pred, zero_division=0
        ),
        "FPR": fp / (fp + tn) if fp + tn else 0
    })

results_df = pd.DataFrame(results)

print(results_df.round(4).to_string(index=False))


In [ ]:
#Error Calculation

import pandas as pd
import numpy as np

# ============================================================
# CONFIGURATION
# ============================================================

model_columns = {
    "Llama-3.1-70B": "LLM_Topic_llama",
    "GPT-5.1": "LLM_Topic_gpt",
    "Gemini-2.5-Flash": "LLM_Topic_gemini",
    "Qwen3-8B": "Qwen_topic"
}

UNASSIGNED = "UNASSIGNED"

df_error = df_human.copy()

def normalize_topic(series):
    return series.astype("string").str.strip().str.upper()

df_error["Human_Normalized"] = normalize_topic(
    df_error["Human_Topic"]
)

# ============================================================
# STEP 1: CLASSIFY ERROR TYPES
# ============================================================

summary_rows = []
error_details = []

for model, column in model_columns.items():

    human = df_error["Human_Normalized"]
    pred = normalize_topic(df_error[column])

    valid = human.notna() & pred.notna()

    h = human[valid]
    p = pred[valid]

    correct_unassigned = (
        (h == UNASSIGNED) & (p == UNASSIGNED)
    )

    false_positive = (
        (h == UNASSIGNED) & (p != UNASSIGNED)
    )

    false_negative = (
        (h != UNASSIGNED) & (p == UNASSIGNED)
    )

    correct_topic = (
        (h != UNASSIGNED) & (p == h)
    )

    wrong_topic = (
        (h != UNASSIGNED) &
        (p != UNASSIGNED) &
        (p != h)
    )

    summary_rows.append({
        "Model": model,
        "N": len(h),
        "Correct_UNASSIGNED": correct_unassigned.sum(),
        "False_Positive": false_positive.sum(),
        "False_Negative": false_negative.sum(),
        "Wrong_Topic": wrong_topic.sum(),
        "Correct_Topic": correct_topic.sum(),
        "Total_Errors": (
            false_positive.sum() +
            false_negative.sum() +
            wrong_topic.sum()
        )
    })

    types = np.select(
        [
            correct_unassigned,
            false_positive,
            false_negative,
            wrong_topic,
            correct_topic
        ],
        [
            "Correct UNASSIGNED",
            "False Positive",
            "False Negative",
            "Wrong Topic",
            "Correct Topic"
        ],
        default="Other"
    )

    details = df_error.loc[valid].copy()
    details["Model"] = model
    details["Predicted_Topic"] = p.values
    details["Error_Type"] = types

    error_details.append(details)


error_summary = pd.DataFrame(summary_rows)

error_details = pd.concat(
    error_details,
    ignore_index=True
)

print("\nERROR SUMMARY")
print(error_summary.to_string(index=False))

# Verify that all examples fall into one of five groups
assert (
    error_summary[
        [
            "Correct_UNASSIGNED",
            "False_Positive",
            "False_Negative",
            "Wrong_Topic",
            "Correct_Topic"
        ]
    ].sum(axis=1).values
    ==
    error_summary["N"].values
).all()